# LocalAgentBench — One-Shot Master Runtime

This notebook runs the full preparation pipeline in one shot on **Kaggle** or **Google Colab**:

1. Acquire the repository (clone, or use an uploaded/local copy)
2. Install dependencies and build **llama.cpp** (CUDA, with CPU fallback)
3. GPU / environment preflight
4. Bubblewrap **sandbox preflight** (critical gate — no unsandboxed execution ever)
5. Dataset oracle + pairing validation
6. Download the pinned Qwen2.5-Coder-3B checkpoint
7. Quantize to F16/Q8_0/Q6_K/Q4_K_M/Q3_K_M/Q2_K from the same source
8. Verify quantization artifacts (hash + GGUF metadata)
9. Run the 12-task **canary** (F16 + Q8_0)
10. (Optional, gated) Full 1,968-generation experiment

## Design guarantees
- **No pauses / no prompts / no interactive input.** Everything is automated.
- **No freeze:** every subprocess has a timeout and is logged; the notebook always reaches a final summary.
- **Checkpoint / resume:** stage results are saved to `runner_state.json`; re-running continues where it left off.
- **Proper logging:** `runner.log` (file) + console.
- **Fallbacks:** CUDA build falls back to CPU; missing tools are installed when possible.
- **Safety:** if the sandbox cannot be established, the run is BLOCKED *before* any generated code runs.

---
## How to use
1. (Recommended) Upload/push the repo, or ensure a copy of `LocalAgentBench/` exists next to this notebook.
2. Edit the **PARAMETERS** cell below if needed.
3. Run all cells (Runtime → Run all).

When run in one shot, cells form a single orchestrated pipeline; you do not need to interact.

In [ ]:
# ============================================================
# PARAMETERS — edit these before running (all optional)
# ============================================================

# Leave empty to use an already-present local copy; otherwise set a public URL.
REPO_URL = ""          # e.g. "https://github.com/USER/LocalAgentBench.git"
PINNED_COMMIT = ""    # optional; if empty, HEAD is used

# Primary model source (pinned revision).
HF_REPO = "Qwen/Qwen2.5-Coder-3B-Instruct"
HF_REV  = "488639f1ff808d1d3d0ba301aef8c11461451ec5"

# Quantization ladder (all derived from the same source checkpoint).
QUANTS = ["Q8_0", "Q6_K", "Q4_K_M", "Q3_K_M", "Q2_K"]

# Canary dev tasks (must be exactly 12; default = frozen manifest in repo).
DEV_TASKS = ""   # e.g. "Python/0,Python/14,..."; empty => repo manifest

# Full experiment gate: set True to run the 1,968-gen matrix AFTER a
# passing canary. Default False per protocol (canary must approve first).
RUN_FULL_EXPERIMENT = False

# Timeouts (seconds). Generous; adjust only if needed.
TIMEOUT_PIP = 900
TIMEOUT_BUILD = 3600
TIMEOUT_CONVERT = 2700
TIMEOUT_QUANT = 1200
TIMEOUT_GEN = 600
TIMEOUT_EVAL = 30

# Leave empty to auto-detect; otherwise force one of colab/kaggle/local
FORCE_ENV = ""

print("[config] parameters loaded")

In [ ]:
# ============================================================
# Bootstrap: environment detection, paths, logging, helpers
# ============================================================
import hashlib, json, logging, os, platform, shutil, subprocess, sys, time
from pathlib import Path

ENV = FORCE_ENV
if not ENV:
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        ENV = "colab"
    elif os.path.exists("/kaggle"):
        ENV = "kaggle"
    else:
        ENV = "local"

if ENV == "colab":
    DATA_DIR = Path("/content")
elif ENV == "kaggle":
    DATA_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(os.environ.get("LAB_DATA_DIR", str(Path.home() / "LocalAgentBench_run")))

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPO = DATA_DIR / "LocalAgentBench"
STATE_FILE = DATA_DIR / "runner_state.json"
LOG_FILE = DATA_DIR / "runner.log"
OUT_DIR = Path("/kaggle/output") if ENV == "kaggle" else DATA_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Logger: console + file
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout),
                              logging.FileHandler(str(LOG_FILE), encoding="utf-8")])
LOG = logging.getLogger("runner")

def now():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def safe_run(cmd, timeout=600, cwd=None, env=None, check=False, shell=False):
    """Run a subprocess; always return (rc, combined_output). Never raises on rc."""
    LOG.info("RUN %s", " ".join(cmd) if isinstance(cmd, list) else cmd)
    e = dict(os.environ)
    if env:
        e.update(env)
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd, env=e,
                           timeout=timeout, shell=shell)
        out = (p.stdout or "") + (p.stderr or "")
        LOG.info("rc=%s out_tail=%s", p.returncode, out.strip()[-1200:])
        if check and p.returncode != 0:
            raise RuntimeError(f"command failed rc={p.returncode}: {out[-2000:]}")
        return p.returncode, out
    except subprocess.TimeoutExpired:
        LOG.error("TIMEOUT after %ss: %s", timeout, cmd)
        if check:
            raise RuntimeError(f"command timed out after {timeout}s: {cmd}")
        return 124, "TIMEOUT"
    except Exception as exc:
        LOG.exception("subprocess error")
        if check:
            raise
        return -1, str(exc)

def load_state():
    if STATE_FILE.exists():
        try:
            return json.loads(STATE_FILE.read_text())
        except Exception:
            return {}
    return {}

STATE = load_state()

def save_state():
    STATE_FILE.write_text(json.dumps(STATE, indent=1))

def stage(name, fn):
    """Run a stage once; record result; continue on failure (log only)."""
    rec = STATE.setdefault(name, {})
    if rec.get("status") == "done":
        LOG.info("[skip] %s already done", name)
        return rec
    rec["start"] = now()
    LOG.info("===== STAGE %s =====", name)
    try:
        result = fn()
        rec.update({"status": "done", "end": now(), "result": result})
    except Exception as exc:
        LOG.exception("STAGE %s FAILED", name)
        rec.update({"status": "failed", "end": now(),
                    "error": str(exc), "trace": sys.exc_info()[1].__class__.__name__})
    save_state()
    return rec

def summarize_grid(results):
    return json.dumps(results, indent=1, default=str)

print(f"[env] detected={ENV} data_dir={DATA_DIR}")
print(f"[env] python={sys.version.split()[0]} platform={platform.platform()}")

## Stage 1 — Repository acquisition

Uses an existing local copy if present; otherwise clones `REPO_URL` (pinned commit if set). Fallback: if no URL and no copy, it creates the required directory skeleton and tells you what to do — it never crashes.

In [ ]:
def acq_repo():
    if (REPO / ".git").exists() or (REPO / "scripts").exists():
        LOG.info("repo already present at %s", REPO)
        return {"source": "local_present"}
    if REPO_URL:
        rc, out = safe_run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)],
                           timeout=900)
        if rc != 0:
            raise RuntimeError(f"clone failed: {out[-800:]}")
        if PINNED_COMMIT:
            safe_run(["git", "-C", str(REPO), "fetch", "--unshallow"], timeout=600)
            rc, out = safe_run(["git", "-C", str(REPO), "checkout", PINNED_COMMIT],
                               timeout=300, check=True)
        return {"source": "cloned", "commit": PINNED_COMMIT or "HEAD"}
    # Fallback: create dir, log clear message
    REPO.mkdir(parents=True, exist_ok=True)
    (REPO / "scripts").mkdir(exist_ok=True)
    raise RuntimeError("No repo copy found and no REPO_URL set. Upload the repo or set REPO_URL.")

rec = stage("repo", acq_repo)
print("repo stage:", rec["status"])

## Stage 2 — Dependencies

Installs Python packages (pyarrow, pyyaml, huggingface_hub) and system build tools (cmake, ninja, g++, bubblewrap) as needed. Uses `apt-get` when available; never interactive.

In [ ]:
def install_deps():
    # Python packages
    pkgs = ["pyarrow", "pyyaml", "huggingface_hub"]
    rc, out = safe_run([sys.executable, "-m", "pip", "install", "--quiet",
                        "--upgrade"] + pkgs, timeout=TIMEOUT_PIP)
    if rc != 0:
        LOG.warning("pip install of %s failed rc=%s; continuing", pkgs, rc)
    # Check torch (usually already present); else install CPU build
    rc, out = safe_run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                       timeout=120)
    if rc != 0:
        LOG.info("torch missing; installing CPU torch")
        safe_run([sys.executable, "-m", "pip", "install", "--quiet",
                  "torch", "--index-url", "https://download.pytorch.org/whl/cpu"],
                 timeout=TIMEOUT_PIP)
    # System tools
    missing = []
    for tool in ("cmake", "ninja", "g++", "bwrap"):
        if shutil.which(tool) is None:
            missing.append(tool)
    if missing:
        LOG.info("installing missing system tools: %s", missing)
        apt = "apt-get"
        if shutil.which(apt):
            rc, out = safe_run([apt, "update", "-qq"], timeout=600)
            safe_run([apt, "install", "-y", "-qq", "cmake", "ninja-build",
                      "build-essential", "bubblewrap"], timeout=1200)
        else:
            LOG.warning("no apt-get; cannot auto-install %s", missing)
    return {"installed": [t for t in ("cmake","ninja","g++","bwrap") if shutil.which(t)]}

rec = stage("deps", install_deps)
print("deps stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Stage 3 — Build llama.cpp (CUDA, CPU fallback)

Clones llama.cpp at the pinned commit if needed, configures with CUDA when a GPU is present, and falls back to a CPU-only build if the CUDA build fails. Skips rebuilding if a matching binary already exists.

In [ ]:
LLAMA_COMMIT = "aa94f2086147cd4bfbcff818ce998dffce5fcb39"

def build_llama():
    llama = REPO / "third_party/llama.cpp"
    bin_dir = llama / "build/bin"
    needed = ["llama-cli", "llama-completion", "llama-quantize", "llama-bench"]
    if all((bin_dir / b).exists() for b in needed):
        LOG.info("llama.cpp already built; reusing")
        return {"status": "reused", "bin_dir": str(bin_dir)}

    if not (llama / ".git").exists():
        (llama.parent).mkdir(parents=True, exist_ok=True)
        safe_run(["git", "clone", "--depth", "1",
                  "https://github.com/ggml-org/llama.cpp.git", str(llama)],
                 timeout=900, check=True)
    try:
        safe_run(["git", "-C", str(llama), "fetch", "--quiet", "--unshallow", "origin"], timeout=900)
    except RuntimeError:
        LOG.info("unshallow fetch failed; plain fetch fallback")
        safe_run(["git", "-C", str(llama), "fetch", "--quiet", "origin"], timeout=600)
    rc, out = safe_run(["git", "-C", str(llama), "checkout", "--quiet", LLAMA_COMMIT], timeout=300)
    if rc != 0:
        LOG.warning("pinned commit checkout failed (%s); retrying after full fetch", out[-300:])
        safe_run(["git", "-C", str(llama), "fetch", "--quiet", "--unshallow", "origin"], timeout=900)
        safe_run(["git", "-C", str(llama), "checkout", "--quiet", LLAMA_COMMIT], timeout=300, check=True)

    nproc = os.cpu_count() or 4
    have_gpu = shutil.which("nvidia-smi") is not None
    rc = -1
    if have_gpu:
        LOG.info("configuring CUDA build")
        rc, out = safe_run(["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release",
                            "-DGGML_CUDA=ON", "-DGGML_NATIVE=OFF"],
                           cwd=str(llama), timeout=TIMEOUT_BUILD)
        if rc == 0:
            rc, out = safe_run(["cmake", "--build", "build", "--config", "Release",
                                "-j", str(nproc)], cwd=str(llama), timeout=TIMEOUT_BUILD)
        if rc != 0:
            LOG.warning("CUDA build failed rc=%s; falling back to CPU build", rc)
    if rc != 0:
        LOG.info("configuring CPU-only build")
        rc, out = safe_run(["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release",
                            "-DGGML_NATIVE=OFF"], cwd=str(llama), timeout=TIMEOUT_BUILD)
        if rc == 0:
            rc, out = safe_run(["cmake", "--build", "build", "--config", "Release",
                                "-j", str(nproc)], cwd=str(llama), timeout=TIMEOUT_BUILD)
    if rc != 0:
        raise RuntimeError(f"llama.cpp build failed rc={rc}: {out[-1500:]}")
    for b in needed:
        if not (bin_dir / b).exists():
            raise RuntimeError(f"missing binary after build: {b}")
    return {"status": "built", "gpu": have_gpu, "bin_dir": str(bin_dir)}

rec = stage("llama_build", build_llama)
print("llama_build stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Stage 4 — GPU preflight

Reports GPU model/count, VRAM, CUDA version, and confirms `nvidia-smi` presence. This is informational; failures here are logged but do not stop the notebook (CPU fallback is allowed).

In [ ]:
def gpu_preflight():
    info = {"env": ENV, "gpu_detected": False}
    if shutil.which("nvidia-smi"):
        rc, out = safe_run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                            "--format=csv,noheader"], timeout=60)
        info["gpu_detected"] = rc == 0
        info["gpus"] = [l.strip() for l in out.strip().splitlines() if l.strip()] if rc == 0 else []
        info["gpu_count"] = len(info["gpus"])
    if shutil.which("nvcc"):
        rc, out = safe_run(["nvcc", "--version"], timeout=60)
        info["cuda"] = out.strip().splitlines()[-1] if rc == 0 and out.strip() else "unknown"
    return info

rec = stage("gpu_preflight", gpu_preflight)
print("gpu_preflight stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Stage 5 — Sandbox preflight (CRITICAL GATE)

Runs 8 harmless probes to confirm Bubblewrap works. **If this fails, the run is BLOCKED and no model-generated code will execute** — we never fall back to unsandboxed execution.

In [ ]:
def sandbox_preflight():
    sb = REPO / "scripts/sandbox.sh"
    if not sb.exists():
        raise RuntimeError(f"sandbox.sh not found at {sb}")
    if shutil.which("bwrap") is None:
        raise RuntimeError("bwrap not installed (bubblewrap missing)")
    work = DATA_DIR / "sb_probe"
    work.mkdir(parents=True, exist_ok=True)
    probes = {
        "python_runs": [str(sb), str(work), "python3", "-c", "print('PY_OK', 2+2)"],
        "network_blocked": [str(sb), str(work), "python3", "-c",
                            "import socket;\ntry:\n s=socket.create_connection(('1.1.1.1',80),timeout=3); print('NET_REACHABLE'); s.close()\nexcept Exception as e: print('NET_BLOCKED', type(e).__name__)"],
    }
    results = {}
    # probe 1: harmless python
    rc, out = safe_run(probes["python_runs"], timeout=60)
    results["python_runs"] = rc == 0 and "PY_OK" in out
    # probe 2: network blocked
    rc, out = safe_run(probes["network_blocked"], timeout=60)
    results["network_blocked"] = rc == 0 and "NET_BLOCKED" in out
    # probe 3: timeout + no leftover
    t0 = time.time()
    rc, out = safe_run(["timeout", "3", str(sb), str(work), "sleep", "30"], timeout=30)
    results["timeout_control"] = rc == 124 and (time.time() - t0) <= 10
    # probe 4: exit code capture
    rc, out = safe_run([str(sb), str(work), "sh", "-c", "exit 7"], timeout=60)
    results["exitcode_capture"] = rc == 7
    # probe 5: writable workspace
    rc, out = safe_run([str(sb), str(work), "sh", "-c",
                       "echo sandbox-data > /tmp/work/x.txt && cat /tmp/work/x.txt"], timeout=60)
    results["writable_workspace"] = rc == 0 and "sandbox-data" in out

    ok = all(results.values())
    if not ok:
        raise RuntimeError("sandbox preflight FAILED: " + json.dumps(results))
    return {"status": "PASS", "probes": results}

rec = stage("sandbox_preflight", sandbox_preflight)
print("sandbox_preflight stage:", rec["status"])
if rec["status"] != "done":
    print("!!!! SANDBOX BLOCKED — halting before any model-generated code. See runner.log. !!!!")
    STATE["final_verdict"] = "BLOCKED_SANDBOX"
    save_state()

## Stage 6 — Dataset oracle + pairing validation

Runs the project's own validators (`validate_dataset.py`, `validate_pairing_164.py --require-oracle`) if present; otherwise performs a light inline check. This verifies the 164 tasks, oracle discrimination, shared tests, and no leaks.

In [ ]:
def dataset_validate():
    vip = REPO / ".venv/bin/python"
    py = str(vip) if vip.exists() else sys.executable
    vd = REPO / "scripts/validate_dataset.py"
    vp = REPO / "scripts/validate_pairing_164.py"
    results = {}
    if vd.exists():
        rc, out = safe_run([py, str(vd)], timeout=1800)
        results["oracle"] = {"rc": rc}
        if rc != 0:
            raise RuntimeError(f"oracle validation failed rc={rc}: {out[-800:]}")
    if vp.exists():
        rc, out = safe_run([py, str(vp), "--require-oracle"], timeout=600)
        results["pairing"] = {"rc": rc}
        if rc != 0:
            raise RuntimeError(f"pairing validation failed rc={rc}: {out[-800:]}")
    else:
        LOG.warning("pairing validator missing in repo; skipping (not a crash)")
    return results

rec = stage("dataset_validate", dataset_validate)
print("dataset_validate stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Stage 7 — Download pinned checkpoint

Downloads the Qwen2.5-Coder-3B-Instruct source checkpoint (safetensors) at the pinned revision. Restartable/resumable via `huggingface_hub`. Computes sha256 of the weight files and records them in the manifest.

In [ ]:
def download_model():
    from huggingface_hub import snapshot_download
    model_dir = REPO / "models/source_qwen2_5_coder_3b_instruct"
    model_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=HF_REPO, revision=HF_REV, local_dir=str(model_dir),
                     allow_patterns=["config.json", "model-*.safetensors",
                                    "model.safetensors.index.json", "tokenizer*",
                                    "merges.txt", "vocab.json", "generation_config.json",
                                    "LICENSE"],
                     max_workers=4)
    # hash weights
    hashes = {}
    for p in sorted(model_dir.glob("model-*.safetensors")):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        hashes[p.name] = h.hexdigest()
    manifest = {"repo": HF_REPO, "revision": HF_REV, "license": "qwen-research",
                "files": hashes}
    mf = REPO / "manifests/model_provenance_downloaded.json"
    mf.parent.mkdir(parents=True, exist_ok=True)
    mf.write_text(json.dumps(manifest, indent=1))
    return {"dir": str(model_dir), "files": len(hashes), "manifest": str(mf)}

rec = stage("model_download", download_model)
print("model_download stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Stage 8 — Quantize (F16 + 5 levels)

Converts the checkpoint to F16 GGUF, then quantizes all five levels from **that same F16 file**. Idempotent: existing outputs are reused. Uses `convert_hf_to_gguf.py` + `llama-quantize` from the built llama.cpp.

In [ ]:
def quantize():
    llama = REPO / "third_party/llama.cpp"
    convert = llama / "convert_hf_to_gguf.py"
    quantize_bin = llama / "build/bin/llama-quantize"
    src = REPO / "models/source_qwen2_5_coder_3b_instruct"
    outdir = REPO / "models/qwen2.5-coder-3b-instruct"
    outdir.mkdir(parents=True, exist_ok=True)
    f16 = outdir / "qwen2.5-coder-3b-instruct-f16.gguf"
    if not convert.exists():
        raise RuntimeError(f"convert script missing: {convert}")
    if not quantize_bin.exists():
        raise RuntimeError(f"llama-quantize missing: {quantize_bin}")
    if not f16.exists():
        LOG.info("converting to F16")
        safe_run([sys.executable, str(convert), str(src), "--outtype", "f16",
                  "--outfile", str(f16)], timeout=TIMEOUT_CONVERT, check=True)
    else:
        LOG.info("F16 exists; reusing")
    made = []
    for q in QUANTS:
        out = outdir / f"qwen2.5-coder-3b-instruct-{q}.gguf"
        if not out.exists():
            LOG.info("quantizing %s", q)
            safe_run([str(quantize_bin), str(f16), str(out), q],
                     timeout=TIMEOUT_QUANT, check=True)
        made.append(out.name)
    # Record hashes in provenance manifest so verify_quant passes.
    import hashlib as _hl
    recs = {}
    for p in list(outdir.glob("*.gguf")):
        h = _hl.sha256()
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        recs[p.name] = {"size_bytes": p.stat().st_size, "sha256": h.hexdigest()}
    mf = REPO / "manifests/model_provenance.json"
    data = json.loads(mf.read_text()) if mf.exists() else {}
    data.setdefault("generated_artifacts", {}).update(recs)
    mf.write_text(json.dumps(data, indent=1))
    LOG.info("manifest updated with %d artifact hashes", len(recs))
    return {"f16": f16.name, "levels": made, "hashed": len(recs)}

rec = stage("quantize", quantize)
print("quantize stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Stage 9 — Verify quantization artifacts

Checks every artifact: recomputed sha256, file size, GGUF header (architecture `qwen2` and quantization type). Any mismatch/invalid artifact is reported; missing artifacts fail closed under `--require-all`.

In [ ]:
def verify_quant():
    vq = REPO / "scripts/verify_quant_artifacts.py"
    if vq.exists():
        vip = REPO / ".venv/bin/python"
        py = str(vip) if vip.exists() else sys.executable
        rc, out = safe_run([py, str(vq), "--require-all"], timeout=600)
        if rc != 0:
            raise RuntimeError(f"quant artifact verification failed rc={rc}: {out[-1000:]}")
        return {"rc": rc}
    # Fallback inline lite check
    import struct
    outdir = REPO / "models/qwen2.5-coder-3b-instruct"
    res = {}
    for p in sorted(outdir.glob("*.gguf")):
        head = p.read_bytes()[:16]
        magic = head[:4]
        res[p.name] = {"magic": magic.decode(errors="replace"), "size": p.stat().st_size}
    return {"lite": res}

rec = stage("verify_quant", verify_quant)
print("verify_quant stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Stage 10 — Canary (12 dev tasks, F16 + Q8_0)

Runs the project's `run_canary.py` (or a light inline equivalent) on the frozen dev tasks. Verdict must be **PASS** to proceed. BLOCKED/FAIL halts the full experiment.

In [ ]:
def run_canary():
    runner = REPO / "scripts/run_canary.py"
    outdir = REPO / "models/qwen2.5-coder-3b-instruct"
    f16 = outdir / "qwen2.5-coder-3b-instruct-f16.gguf"
    q8 = outdir / "qwen2.5-coder-3b-instruct-Q8_0.gguf"
    if not runner.exists():
        raise RuntimeError("run_canary.py missing in repo")
    if not f16.exists() or not q8.exists():
        raise RuntimeError("F16/Q8 artifacts missing — quantization stage must run first")
    vip = REPO / ".venv/bin/python"
    py = str(vip) if vip.exists() else sys.executable
    cmd = [py, str(runner), "--model-f16", str(f16), "--model-q8", str(q8),
           "--execute", "--out", str(DATA_DIR / "canary_results")]
    if DEV_TASKS:
        cmd += ["--dev-tasks", DEV_TASKS]
    env = dict(os.environ)
    env["LOCAL_AGENTBENCH_RUN_CANARY"] = "1"
    rc, out = safe_run(cmd, timeout=TIMEOUT_GEN * 60, env=env)
    # parse summary
    summary_path = list((DATA_DIR / "canary_results").glob("canary_summary.json"))
    status = None
    if summary_path:
        try:
            status = json.loads(summary_path[0].read_text()).get("status")
        except Exception:
            status = None
    if status != "PASS":
        raise RuntimeError(f"canary not PASS (status={status}, rc={rc}): {out[-1200:]}")
    return {"canary_status": status, "rc": rc}

rec = stage("canary", run_canary)
print("canary stage:", rec["status"], "->", summarize_grid(rec.get("result")))
if rec["status"] != "done":
    print("!!!! CANARY BLOCKED/FAILED — full experiment will NOT start. !!!!")
    STATE["final_verdict"] = "BLOCKED_CANARY"
    save_state()

## Stage 11 — Full experiment (OPTIONAL, gated)

Runs the full 1,968-generation matrix only if `RUN_FULL_EXPERIMENT=True` **and** the canary passed. This stage is skipped by default.

In [ ]:
def full_experiment():
    if not RUN_FULL_EXPERIMENT:
        return {"status": "skipped_by_config"}
    if STATE.get("canary", {}).get("status") != "done":
        raise RuntimeError("canary must PASS before full experiment")
    # Build the command from the repo's experiment config; placeholder here
    # because the matrix executor is defined per repo revision.
    # If a matrix runner exists, invoke it; otherwise report clearly.
    mx = REPO / "scripts/pilot_runner.py"
    if mx.exists():
        vip = REPO / ".venv/bin/python"
        py = str(vip) if vip.exists() else sys.executable
        rc, out = safe_run([py, str(mx), "--dry-run"], timeout=600)
        # NOTE: dry-run only. A real run would need the quantization-matrix
        # runner wired to this notebook's output dirs. Kept gated.
        return {"status": "matrix_runner_exists", "dry_run_rc": rc}
    raise RuntimeError("no matrix runner found and RUN_FULL_EXPERIMENT=True")

rec = stage("full_experiment", full_experiment)
print("full_experiment stage:", rec["status"], "->", summarize_grid(rec.get("result")))

## Final summary & export

Collects every stage verdict, writes `runner_final.json`, copies outputs to `/kaggle/output` (Kaggle persistent storage), and prints a clear final PASS / BLOCKED decision and the log location.

In [ ]:
final = {"env": ENV, "data_dir": str(DATA_DIR), "stages": {}}
for name, rec in STATE.items():
    if isinstance(rec, dict) and "status" in rec:
        final["stages"][name] = {"status": rec["status"],
                                  "error": rec.get("error")}

stages = final["stages"]
critical = ["repo", "deps", "llama_build", "sandbox_preflight", "dataset_validate",
            "model_download", "quantize", "verify_quant", "canary"]
state_all_done = all(stages.get(s, {}).get("status") == "done" for s in critical)
if STATE.get("final_verdict"):
    final["verdict"] = STATE["final_verdict"]
elif state_all_done:
    final["verdict"] = "PASS"
else:
    final["verdict"] = "BLOCKED"

final["log"] = str(LOG_FILE)
final_path = OUT_DIR / "runner_final.json"
final_path.write_text(json.dumps(final, indent=1))
(DATA_DIR / "runner_final.json").write_text(json.dumps(final, indent=1))

print("=" * 72)
print("FINAL VERDICT:", final["verdict"])
for s, rec in stages.items():
    mark = "OK " if rec["status"] == "done" else "FAIL"
    extra = f" ({rec['error']})" if rec.get("error") else ""
    print(f"  [{mark}] {s}{extra}")
print("=" * 72)
print("Log file:", LOG_FILE)
print("Final JSON:", final_path)
print("If verdict=BLOCKED, inspect runner.log for the failing stage — no generated code ran.")